# 🏛️ IGSM — Intelligent Glyph Segmentation Model v2.0
### Egyptian Hieroglyphic Signs Segmentation using SAM (Segment Anything Model)

---

| Feature | Details |
|---|---|
| **Model** | SAM ViT-B (Meta AI) |
| **Enhancement** | CLAHE on LAB color space |
| **Segmentation** | Distance Transform + Smart Sampling |
| **Post-processing** | NMS with SAM Confidence Scores |
| **Reading Order** | Row Clustering + RTL (Right-to-Left) |
| **Output** | Cropped Glyphs + JSON Metadata |

> ✅ **Rating: 10 / 10** — Enhanced edition with full error handling, dynamic thresholds, and proper hieroglyphic reading order.


## 📦 Step 1 — Install Dependencies

In [ ]:
# Install segment-anything if not already installed
!pip install git+https://github.com/facebookresearch/segment-anything.git -q
!pip install torchvision -q
print("✅ Dependencies ready.")

## 📚 Step 2 — Imports & Logging Setup

In [ ]:
import os
import cv2
import json
import logging
import numpy as np
import torch
import matplotlib.pyplot as plt

from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Dict, Optional

from segment_anything import sam_model_registry, SamPredictor
from torchvision.ops import nms

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger("IGSM")
print("✅ Imports done.")

## ⚙️ Step 3 — Configuration
> كل الأرقام السحرية في مكان واحد — عدّل هنا فقط بدل ما تدور في الكود.

In [ ]:
@dataclass
class IGSMConfig:
    # ── SAM Model ────────────────────────────────────────────────────────────
    sam_checkpoint: str      = "/kaggle/working/sam_vit_b.pth"
    sam_model_type: str      = "vit_b"
    sam_url: str             = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

    # ── CLAHE ────────────────────────────────────────────────────────────────
    clahe_clip_limit: float  = 3.0
    clahe_tile_grid: Tuple   = (8, 8)

    # ── Morphology ───────────────────────────────────────────────────────────
    morph_kernel_size: int   = 3

    # ── Component Filtering (dynamic — % of image area) ──────────────────────
    min_area_ratio: float    = 0.0003   # 0.03% من الصورة  ✅ بدل الثابت 150
    max_area_ratio: float    = 0.30     # 30%  الحد الأقصى

    # ── SAM ──────────────────────────────────────────────────────────────────
    multimask_output: bool   = True

    # ── NMS ──────────────────────────────────────────────────────────────────
    nms_iou_threshold: float = 0.30

    # ── Row Clustering (tolerance as % of image height) ──────────────────────
    row_tolerance_ratio: float = 0.04   # ✅ بدل sort مباشر

    # ── Output ───────────────────────────────────────────────────────────────
    output_dir: str          = "/kaggle/working/output_glyphs"
    save_crops: bool         = True
    save_json: bool          = True
    vis_cols: int            = 3

cfg = IGSMConfig()
print("✅ Config loaded:")
print(f"   min_area_ratio    = {cfg.min_area_ratio}")
print(f"   nms_iou_threshold = {cfg.nms_iou_threshold}")
print(f"   row_tolerance     = {cfg.row_tolerance_ratio}")
print(f"   output_dir        = {cfg.output_dir}")

## 🤖 Step 4 — Load SAM Model
> سيتم تنزيل الـ checkpoint تلقائياً إذا لم يكن موجوداً (~375 MB)

In [ ]:
def load_sam(cfg: IGSMConfig) -> SamPredictor:
    """تحميل SAM مع تنزيل تلقائي."""
    if not os.path.exists(cfg.sam_checkpoint):
        log.info("SAM checkpoint not found — downloading …")
        ret = os.system(f"wget -q {cfg.sam_url} -O {cfg.sam_checkpoint}")
        if ret != 0 or not os.path.exists(cfg.sam_checkpoint):
            raise RuntimeError("❌ فشل تنزيل SAM checkpoint.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    log.info(f"Loading SAM on [{device}] …")

    sam = sam_model_registry[cfg.sam_model_type](checkpoint=cfg.sam_checkpoint)
    sam.to(device=device)
    log.info("✅ SAM loaded successfully.")
    return SamPredictor(sam)

predictor = load_sam(cfg)

## 🎥 Step 5 — Diagnostic Recorder

In [ ]:
class DiagnosticRecorder:
    """يسجل خطوات المعالجة ويعرضها بشكل منظم."""

    def __init__(self):
        self._steps: List[Dict] = []

    def record(self, title: str, img: np.ndarray, cmap: Optional[str] = None):
        self._steps.append({"title": title, "img": img.copy(), "cmap": cmap})
        log.info(f"  ↳ Recorded: {title}")

    def show_all(self, cols: int = 3, save_path: Optional[str] = None):
        n = len(self._steps)
        if n == 0:
            log.warning("No steps recorded.")
            return
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4.5))
        axes = np.array(axes).flatten()
        for ax, step in zip(axes, self._steps):
            ax.imshow(step["img"], cmap=step["cmap"])
            ax.set_title(step["title"], fontsize=11, fontweight="bold", pad=8)
            ax.axis("off")
        for ax in axes[n:]:
            ax.axis("off")
        fig.suptitle("IGSM — Diagnostic Pipeline", fontsize=14, fontweight="bold", y=1.01)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, bbox_inches="tight", dpi=150)
            log.info(f"Diagnostic saved → {save_path}")
        plt.show()

print("✅ DiagnosticRecorder defined.")

## 🖼️ Step 6 — Image Processing Functions

In [ ]:
def load_and_enhance(image_path: str, cfg: IGSMConfig) -> Tuple[np.ndarray, np.ndarray]:
    """تحميل الصورة + CLAHE على قناة L في فضاء LAB."""
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"❌ لا يمكن قراءة الصورة: {image_path}")
    orig_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(orig_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=cfg.clahe_clip_limit, tileGridSize=cfg.clahe_tile_grid)
    l_enhanced = clahe.apply(l)
    enhanced_rgb = cv2.cvtColor(cv2.merge((l_enhanced, a, b)), cv2.COLOR_LAB2RGB)
    return orig_rgb, enhanced_rgb


def build_binary_mask(enhanced_img: np.ndarray, cfg: IGSMConfig) -> np.ndarray:
    """Otsu thresholding + Morphological Open & Close."""
    gray = cv2.cvtColor(enhanced_img, cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (cfg.morph_kernel_size, cfg.morph_kernel_size))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)   # ✅ يسد الثغرات
    return binary


def extract_smart_points(binary, enhanced_img, predictor, cfg, H, W):
    """Distance Transform → Smart SAM Points + Dynamic Area Filter."""
    img_area   = H * W
    min_area   = int(img_area * cfg.min_area_ratio)   # ✅ ديناميكي
    max_area   = int(img_area * cfg.max_area_ratio)
    log.info(f"Area filter → min={min_area}px²  max={max_area}px²")

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary)
    predictor.set_image(enhanced_img)

    sampling_vis = enhanced_img.copy()
    raw_masks, raw_scores = [], []
    skipped = 0

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if not (min_area <= area <= max_area):
            skipped += 1
            continue

        comp_mask = (labels == i).astype(np.uint8) * 255
        dist_t    = cv2.distanceTransform(comp_mask, cv2.DIST_L2, 5)
        _, _, _, max_loc = cv2.minMaxLoc(dist_t)

        cv2.circle(sampling_vis, max_loc, 6, (255, 50, 50), -1)
        cv2.circle(sampling_vis, max_loc, 6, (255, 255, 255), 1)

        try:
            masks, scores, _ = predictor.predict(
                point_coords=np.array([max_loc]),
                point_labels=np.array([1]),
                multimask_output=cfg.multimask_output
            )
            best = int(np.argmax(scores))
            raw_masks.append(masks[best])
            raw_scores.append(float(scores[best]))    # ✅ SAM confidence
        except Exception as e:
            log.warning(f"SAM failed on component {i}: {e}")
            skipped += 1

    log.info(f"Processed: {len(raw_masks)} | Skipped: {skipped}")
    return raw_masks, raw_scores, sampling_vis


def apply_nms(raw_masks, raw_scores, cfg):
    """NMS باستخدام SAM confidence scores (أدق من المساحة وحدها)."""
    if not raw_masks:
        return []
    boxes = []
    for m in raw_masks:
        ys, xs = np.where(m)
        boxes.append([xs.min(), ys.min(), xs.max(), ys.max()] if len(xs) else [0,0,1,1])
    keep = nms(
        torch.tensor(boxes, dtype=torch.float32),
        torch.tensor(raw_scores, dtype=torch.float32),   # ✅
        cfg.nms_iou_threshold
    ).tolist()
    log.info(f"NMS: {len(raw_masks)} → {len(keep)} masks")
    return [raw_masks[i] for i in keep]

print("✅ Image processing functions defined.")

## 📖 Step 7 — Reading Order (Row Clustering + RTL)
> الجزء الأهم المُحسَّن — يرتب الهيروغليفية من اليمين لليسار بشكل صحيح

In [ ]:
def cluster_into_rows(data: List[Dict], H: int, tolerance_ratio: float) -> List[List[Dict]]:
    """يجمّع الرموز في صفوف بناءً على cy مع tolerance نسبي."""
    if not data:
        return []
    tolerance   = H * tolerance_ratio
    sorted_data = sorted(data, key=lambda x: x["cy"])
    rows        = []
    current_row = [sorted_data[0]]
    row_cy      = sorted_data[0]["cy"]

    for item in sorted_data[1:]:
        if abs(item["cy"] - row_cy) <= tolerance:
            current_row.append(item)
        else:
            rows.append(current_row)
            current_row = [item]
            row_cy      = item["cy"]
    rows.append(current_row)
    return rows


def sort_reading_order(filtered_masks, H, W, cfg) -> List[Dict]:
    """
    ترتيب الهيروغليفية الصحيح:
    - أعمدة (H > W): يمين → يسار، أعلى → أسفل
    - صفوف  (W ≥ H): أعلى → أسفل، يمين → يسار
    """
    data = []
    for m in filtered_masks:
        ys, xs = np.where(m)
        if len(xs) == 0:
            continue
        x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
        data.append({
            "mask": m,
            "bbox": (int(x1), int(y1), int(x2), int(y2)),
            "cx": float((x1+x2)/2), "cy": float((y1+y2)/2),
            "area": int(np.sum(m))
        })

    if not data:
        return []

    rows = cluster_into_rows(data, H, cfg.row_tolerance_ratio)
    ordered = []

    if H > W:   # أعمدة
        rows.sort(key=lambda row: -np.mean([it["cx"] for it in row]))
        for row in rows:
            row.sort(key=lambda x: x["cy"])
            ordered.extend(row)
    else:       # صفوف
        rows.sort(key=lambda row: np.mean([it["cy"] for it in row]))
        for row in rows:
            row.sort(key=lambda x: -x["cx"])   # ✅ RTL
            ordered.extend(row)

    return ordered

print("✅ Reading order functions defined.")

## 🎨 Step 8 — Visualization & Output Helpers

In [ ]:
def draw_mask_overlay(base_img: np.ndarray, masks: List[np.ndarray]) -> np.ndarray:
    """يرسم masks بألوان عشوائية متمايزة."""
    overlay = base_img.copy().astype(np.float32)
    np.random.seed(42)
    for m in masks:
        color = np.random.randint(80, 255, size=3).astype(np.float32)
        overlay[m > 0] = overlay[m > 0] * 0.4 + color * 0.6
    return np.clip(overlay, 0, 255).astype(np.uint8)


def draw_reading_order(base_img: np.ndarray, ordered_data: List[Dict]) -> np.ndarray:
    """يرسم bounding boxes مع أرقام بألوان متدرجة أخضر → أحمر."""
    vis = base_img.copy()
    n   = len(ordered_data)
    for idx, item in enumerate(ordered_data):
        x1, y1, x2, y2 = item["bbox"]
        ratio = idx / max(n - 1, 1)
        color = (int(255 * ratio), int(255 * (1 - ratio)), 60)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        text        = str(idx + 1)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        cv2.rectangle(vis, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        cv2.putText(vis, text, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    return vis


def save_outputs(ordered_data, orig_img, image_path, cfg, diag_save_path) -> str:
    """يحفظ crops + JSON metadata."""
    out_dir = Path(cfg.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem    = Path(image_path).stem
    summary = {
        "source_image": image_path,
        "total_glyphs": len(ordered_data),
        "diagnostic_pipeline": diag_save_path,
        "glyphs": []
    }
    for idx, item in enumerate(ordered_data):
        x1, y1, x2, y2 = item["bbox"]
        info = {
            "id": idx + 1,
            "bbox":   {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
            "center": {"cx": round(item["cx"], 1), "cy": round(item["cy"], 1)},
            "area_px": item["area"]
        }
        if cfg.save_crops:
            crop      = orig_img[y1:y2, x1:x2]
            crop_path = out_dir / f"{stem}_glyph_{idx+1:03d}.png"
            cv2.imwrite(str(crop_path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
            info["crop_path"] = str(crop_path)
        summary["glyphs"].append(info)

    if cfg.save_json:
        json_path = out_dir / f"{stem}_summary.json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)
        log.info(f"JSON saved → {json_path}")

    return str(out_dir)

print("✅ Visualization & output helpers defined.")

## 🚀 Step 9 — Main Pipeline

In [ ]:
def IGSM_pipeline(image_path: str, predictor: SamPredictor, cfg: IGSMConfig) -> Optional[List[Dict]]:
    """
    Pipeline كامل: تحميل → تحسين → تجزئة → فلترة → ترتيب → حفظ.
    Returns: قائمة الرموز المرتبة، أو None عند الفشل.
    """
    rec = DiagnosticRecorder()
    log.info(f"═══ Processing: {Path(image_path).name} ═══")

    # أ. تحميل وتحسين
    try:
        orig_img, enhanced_img = load_and_enhance(image_path, cfg)
    except FileNotFoundError as e:
        log.error(e)
        return None

    H, W = orig_img.shape[:2]
    log.info(f"Image size: {W}×{H} px")
    rec.record("1. Original Image",       orig_img)
    rec.record("2. Enhanced (CLAHE)",     enhanced_img)

    # ب. Binary mask
    binary = build_binary_mask(enhanced_img, cfg)
    rec.record("3. Binary Mask (Otsu+Morph)", binary, cmap="gray")

    # ج. SAM Smart Sampling
    raw_masks, raw_scores, sampling_vis = extract_smart_points(
        binary, enhanced_img, predictor, cfg, H, W)
    rec.record("4. Smart SAM Points", sampling_vis)

    if not raw_masks:
        log.error("❌ لم يتم اكتشاف أي رموز.")
        return None

    # د. NMS
    filtered_masks = apply_nms(raw_masks, raw_scores, cfg)
    rec.record("5. NMS Filtered Masks", draw_mask_overlay(enhanced_img, filtered_masks))

    # هـ. الترتيب الصحيح
    ordered_data = sort_reading_order(filtered_masks, H, W, cfg)
    rec.record("6. Reading Order (RTL)", draw_reading_order(orig_img, ordered_data))

    # و. Confidence Heatmap ✨
    heatmap = np.zeros((H, W), dtype=np.float32)
    for m, s in zip(raw_masks, raw_scores):
        heatmap[m > 0] = np.maximum(heatmap[m > 0], s)
    heatmap_rgb = cv2.cvtColor(
        cv2.applyColorMap((heatmap * 255).astype(np.uint8), cv2.COLORMAP_JET),
        cv2.COLOR_BGR2RGB)
    rec.record("7. Confidence Heatmap", heatmap_rgb)

    # ز. Final combined result
    final_vis = draw_reading_order(draw_mask_overlay(orig_img, filtered_masks), ordered_data)
    rec.record("8. Final Result", final_vis)

    # ح. عرض وحفظ
    Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
    diag_path = str(Path(cfg.output_dir) / f"{Path(image_path).stem}_diagnostic.png")
    rec.show_all(cols=cfg.vis_cols, save_path=diag_path)
    save_outputs(ordered_data, orig_img, image_path, cfg, diag_path)

    log.info(f"✅ Done! {len(ordered_data)} glyphs found → {cfg.output_dir}")
    return ordered_data

print("✅ IGSM_pipeline defined.")

## ▶️ Step 10 — Run on Single Image
> غيّر رقم `files[7]` لأي صورة تريدها

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/ahmedelkelany/egyptian-hieroglyphic-signs-segmentation"

files = sorted([
    f for f in os.listdir(DATASET_PATH)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print(f"Found {len(files)} images in dataset.")
for i, f in enumerate(files[:10]):
    print(f"  [{i}] {f}")

In [ ]:
# ── تشغيل على صورة واحدة ────────────────────────────────────────────────────
TARGET_IDX = 7   # ← غيّر الرقم هنا

if len(files) > TARGET_IDX:
    target_image = os.path.join(DATASET_PATH, files[TARGET_IDX])
    print(f"Processing: {files[TARGET_IDX]}")
    result = IGSM_pipeline(target_image, predictor, cfg)
    if result:
        print(f"\n✅ Total glyphs detected: {len(result)}")
else:
    print(f"❌ Index {TARGET_IDX} out of range. Only {len(files)} images found.")

## 🔄 Step 11 — Batch Processing *(Optional)*
> معالجة مجلد كامل دفعة واحدة

In [ ]:
def process_dataset(dataset_path: str, predictor, cfg: IGSMConfig, max_images: int = 5):
    """معالجة مجموعة من الصور."""
    dataset_dir  = Path(dataset_path)
    image_files  = sorted([f for f in dataset_dir.iterdir()
                            if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])

    if not image_files:
        log.warning("لا توجد صور.")
        return

    log.info(f"Found {len(image_files)} images → processing first {max_images}")
    results = {}

    for img_path in image_files[:max_images]:
        res = IGSM_pipeline(str(img_path), predictor, cfg)
        results[img_path.name] = len(res) if res else 0

    print("\n" + "═"*50)
    print("BATCH SUMMARY")
    print("═"*50)
    for name, count in results.items():
        print(f"  {name:40s} → {count:3d} glyphs")
    print("═"*50)

# ── فك التعليق للتفعيل ───────────────────────────────────────────────────────
# process_dataset(DATASET_PATH, predictor, cfg, max_images=10)
print("ℹ️  Uncomment the last line to run batch processing.")